In [1]:
import os
import re
import glob
import pandas as pd
#from matplotlib import pyplot as plt
#import seaborn as sns
#%matplotlib inline
import warnings
warnings.filterwarnings('ignore')
#import plotly.express as px
import numpy as np
#import panel as pn
#import hvplot.pandas
from collections import Counter

In [3]:
#Define path files
path = 'Path_to_folder'
path_fasta = 'path_to_fasta_files'

## Check same number of cells between Alpha and Beta

In [4]:
#Number cells in fasta A and B
def count_unique_names(filename):
    unique_names = set()
    with open(filename, 'r') as file:
        for line in file:
            if line.startswith('>'):
                # Split the line by '.' and extract the second part (index 1)
                name = line.strip().split('.')[1]
                unique_names.add(name)
    return len(unique_names), unique_names


#Number lines fasta and table
def count_lines_starting_with_character(filename):
    count = 0
    with open(filename, 'r') as file:
        for line in file:
            if line.startswith('>'):
                count += 1
    return count

def validate_files(fasta_A, HVQ_A, fasta_B, HVQ_B, IGTname):
    num_lines_fasta_file_A = count_lines_starting_with_character(fasta_A)
    num_rows_in_table_A = len(HVQ_A)

    num_lines_fasta_file_B = count_lines_starting_with_character(fasta_B)
    num_rows_in_table_B = len(HVQ_B)


    Nb_cell_A, unique_names_A = count_unique_names(fasta_A)
    Nb_cell_B, unique_names_B = count_unique_names(fasta_B)
    
    # Find the intersection of unique names
    shared_unique_names = unique_names_A.intersection(unique_names_B)
    num_shared_unique_names = len(shared_unique_names)

    # Create a dictionary for the data
    data = {
        "Info": [
            "Number of cells alpha", 
            "Number of cells beta", 
            "Number of shared cells", 
            "Number of sequences in fasta A", 
            "Number of rows in table A", 
            "Number of sequences in fasta B", 
            "Number of rows in table B"
        ],
        "Value": [
            Nb_cell_A, 
            Nb_cell_B, 
            num_shared_unique_names, 
            num_lines_fasta_file_A, 
            num_rows_in_table_A, 
            num_lines_fasta_file_B, 
            num_rows_in_table_B
        ]
    }

    # Create the DataFrame
    checkfile = pd.DataFrame(data)

    # Define the output file path
    output_file = os.path.join(path, IGTname, f'{IGTname}_output.txt')

    # Save the output to a text file
    checkfile.to_csv(output_file, sep='\t', index=False)

## Format name and types data

In [5]:
def the_cell_name(value):
    new_value = value.split("_V")[0]
    return new_value

In [6]:
def format_gene_name(value):
    # For example for a int
    new_value = value.split(" ")[1]
    return new_value

In [7]:
def format_gene_name_all_suggested(value):
    if pd.isna(value):  # Check for NaN values
        return value
    new_value = str(value)  # Ensure the value is a string
    new_value = new_value.replace("Musmus ","")
    new_value = new_value.replace("F,","")
    new_value = new_value.replace(" F","")
    new_value = new_value.replace(" ORF","")
    new_value = new_value.replace(" P","")
    new_value = new_value.replace(" OR or","|")
    new_value = new_value.replace(" or","|")
    return new_value

In [8]:
def extract_gene_name(value):
    if pd.isna(value):  # Check for NaN values
        return value
    new_value = str(value)  # Ensure the value is a string
    # Replace any number after * with nothing
    new_value = re.sub(r'\*\d+\b', "", value)
    return new_value

### extract name genes

In [9]:
def process_alpha(HVQ_A):
    TRA_HVQ = HVQ_A[['Sequence ID','V-DOMAIN Functionality', 'V-GENE and allele','J-GENE and allele','AA JUNCTION']].copy()
    TRA_HVQ['Sequence ID'] = TRA_HVQ['Sequence ID'].apply(the_cell_name)
    TRA_HVQ['V-GENE and allele'] = TRA_HVQ['V-GENE and allele'].apply(format_gene_name_all_suggested)
    TRA_HVQ['J-GENE and allele'] = TRA_HVQ['J-GENE and allele'].apply(format_gene_name_all_suggested)
    TRA_HVQ['V-GENE'] = TRA_HVQ['V-GENE and allele'].apply(extract_gene_name)
    TRA_HVQ['J-GENE'] = TRA_HVQ['J-GENE and allele'].apply(extract_gene_name)
    return TRA_HVQ

def process_beta(HVQ_B):
    TRB_HVQ = HVQ_B[['Sequence ID','V-DOMAIN Functionality', 'V-GENE and allele','J-GENE and allele','AA JUNCTION']].copy()
    TRB_HVQ['Sequence ID'] = TRB_HVQ['Sequence ID'].apply(the_cell_name)
    TRB_HVQ['V-GENE and allele'] = TRB_HVQ['V-GENE and allele'].apply(format_gene_name_all_suggested)
    TRB_HVQ['J-GENE and allele'] = TRB_HVQ['J-GENE and allele'].apply(format_gene_name_all_suggested)
    TRB_HVQ['V-GENE'] = TRB_HVQ['V-GENE and allele'].apply(extract_gene_name)
    TRB_HVQ['J-GENE'] = TRB_HVQ['J-GENE and allele'].apply(extract_gene_name)
    return TRB_HVQ

### Check number D-Gene for beta

In [10]:
# Check if no D-Gene
def check_D_gene_null(value):
    if pd.isna(value['D-GENE and allele']) or value['D-GENE and allele'] == '':
        #value['beta.vj.nregion'] = value['P3\'V'] + value['N-REGION'] + value['P5\'J']  #with palindrome
        value['beta.vj.nregion'] = value['N-REGION']
    else:
        #value['beta.vd.nregion'] = value['P3\'V'] + value['N1-REGION'] + value['P5\'D'] #with palindrome
        #value['beta.dj.nregion'] = value['P3\'D'] + value['N2-REGION'] + value['P5\'J'] #with palindrome
        value['beta.vd.nregion'] = value['N1-REGION']
        value['beta.dj.nregion'] = value['N2-REGION']
    return value


#def extract_Gene_severalD(value):
#    if pd.notnull(value['D1-REGION']):
#        rows_several_Dgene.append(value['Sequence ID'])
    
#    return rows_several_Dgene

### Add junction information

In [11]:
def process_data_TRA(df, junction_df, junction_columns, merge_columns):
    new_df = junction_df[junction_columns].copy().fillna('')
    new_df['Sequence ID'] = new_df['Sequence ID'].apply(the_cell_name)
    new_df[junction_columns[1:]] = new_df[junction_columns[1:]].apply(lambda x: x.str.upper())
    new_df = new_df.rename(columns={'N-REGION': 'alpha.vj.nregion'})
    merged_df = pd.merge(df, new_df[merge_columns], on='Sequence ID', how='left').fillna('')
    return merged_df

def process_data_TRB(df, junction_df, junction_columns, merge_columns):
    new_df = junction_df[junction_columns].copy().fillna('')
    new_df['Sequence ID'] = new_df['Sequence ID'].apply(the_cell_name)
    new_df[junction_columns[2:]] = new_df[junction_columns[2:]].apply(lambda x: x.str.upper())
    new_df = new_df.apply(check_D_gene_null, axis=1).fillna('')
    merged_df = pd.merge(df, new_df[merge_columns], on='Sequence ID', how='left').fillna('')
    return merged_df

### Merge files to have cgt 1 and 2 for each chain

In [12]:
def gene_name_ctg(value):
    # For example for a int
    new_value = value.split(".ctg")[0]
    return new_value

In [13]:
def process_and_merge_TRA(data, IGTname):
    merged1 = data[data['Sequence ID'].str.endswith('.ctg1')]
    merged2 = data[data['Sequence ID'].str.endswith('.ctg2')]
    merged1['Sequence ID'] = merged1['Sequence ID'].apply(gene_name_ctg)
    merged2['Sequence ID'] = merged2['Sequence ID'].apply(gene_name_ctg)
    
    merged1 = merged1[['Sequence ID','V-DOMAIN Functionality', 'V-GENE','J-GENE','AA JUNCTION', 'JUNCTION', 'alpha.vj.nregion']].copy()
    merged2 = merged2[['Sequence ID','V-DOMAIN Functionality', 'V-GENE', 'J-GENE','AA JUNCTION', 'JUNCTION', 'alpha.vj.nregion']].copy()
    
    merged1 = merged1.rename(columns={'Sequence ID': 'IGT.cellID','V-DOMAIN Functionality': 'alpha.functionality', 'V-GENE': 'alpha.vgene'\
                            ,'J-GENE': 'alpha.jgene','AA JUNCTION': 'alpha.junction', 'JUNCTION': 'alpha.junction.nt', 'alpha.vj.nregion': 'alpha.vj.nregion'}).copy()
    merged2 = merged2.rename(columns={'Sequence ID': 'IGT.cellID','V-DOMAIN Functionality': 'alpha2.functionality', 'V-GENE': 'alpha2.vgene'\
                            ,'J-GENE': 'alpha2.jgene','AA JUNCTION': 'alpha2.junction', 'JUNCTION': 'alpha2.junction.nt', 'alpha.vj.nregion': 'alpha2.vj.nregion'}).copy()
        
    merged = pd.merge(merged1, merged2, on='IGT.cellID', how='left').copy().fillna('')
    print(f"Number of rows in {IGTname} TRA: {len(merged)}")

    return merged

def process_and_merge_TRB(data, IGTname):
    merged1 = data[data['Sequence ID'].str.endswith('.ctg1')]
    merged2 = data[data['Sequence ID'].str.endswith('.ctg2')]
    merged1['Sequence ID'] = merged1['Sequence ID'].apply(gene_name_ctg)
    merged2['Sequence ID'] = merged2['Sequence ID'].apply(gene_name_ctg)
    
    merged1 = merged1[['Sequence ID','V-DOMAIN Functionality', 'V-GENE', 'J-GENE','AA JUNCTION', 'JUNCTION', 'beta.vd.nregion', 'beta.dj.nregion']].copy()
    merged2 = merged2[['Sequence ID','V-DOMAIN Functionality', 'V-GENE', 'J-GENE','AA JUNCTION', 'JUNCTION', 'beta.vd.nregion', 'beta.dj.nregion']].copy()
    
    merged1 = merged1.rename(columns={'Sequence ID': 'IGT.cellID','V-DOMAIN Functionality': 'beta.functionality', 'V-GENE': 'beta.vgene'\
                            ,'J-GENE': 'beta.jgene','AA JUNCTION': 'beta.junction', 'JUNCTION': 'beta.junction.nt'}).copy()
    merged2 = merged2.rename(columns={'Sequence ID': 'IGT.cellID','V-DOMAIN Functionality': 'beta2.functionality', 'V-GENE': 'beta2.vgene'\
                            ,'J-GENE': 'beta2.jgene','AA JUNCTION': 'beta2.junction', 'JUNCTION': 'beta2.junction.nt', 'beta.dj.nregion':'beta2.dj.nregion', 'beta.vd.nregion': 'beta2.vd.nregion'}).copy()
    
    merged = pd.merge(merged1, merged2, on='IGT.cellID', how='left').copy().fillna('')
    print(f"Number of rows in {IGTname} TRB: {len(merged)}")

    return merged

In [14]:
def merge_TRA_TRB(data1, data2, IGTname):
    final = pd.merge(data1, data2, on='IGT.cellID', how='outer').copy()

    final.to_csv(f'{path}{IGTname}/{IGTname}_summary.csv', index=False)

In [17]:
def process_files(path):
   
    # List all the IGT folders
    igt_list = [file for file in os.listdir(path) if file.startswith('IGT') and file != 'IGT43']
    #igt_folders = [folder for folder in os.listdir(path) if folder.startswith('IGT')] #and not folder.startswith('IGT47')]
    igt_folders = [folder for folder in os.listdir(path) if folder in igt_list]
    
    for IGTname in igt_folders:
        path_files_A = glob.glob(f'{path}{IGTname}/{IGTname}_alpha*')
        path_files_B = glob.glob(f'{path}{IGTname}/{IGTname}_beta*')
        # Read the files for each IGT folder
        fasta_A = f'{path_fasta}{IGTname}/{IGTname}_allCells_productiveANDnonproductiveTCR_pairedANDunpaired_trv_cdr3alpha.fasta'
        fasta_B = f'{path_fasta}{IGTname}/{IGTname}_allCells_productiveANDnonproductiveTCR_pairedANDunpaired_trv_cdr3beta.fasta'
        HVQ_A = pd.read_csv(f'{path_files_A[0]}/1_Summary.txt', delimiter="\t", low_memory=False)
        HVQ_B = pd.read_csv(f'{path_files_B[0]}/1_Summary.txt', delimiter="\t", low_memory=False)
        HVQ_A_junction = pd.read_csv(f'{path_files_A[0]}/6_Junction.txt', delimiter="\t", low_memory=False)
        HVQ_B_junction = pd.read_csv(f'{path_files_B[0]}/6_Junction.txt', delimiter="\t", low_memory=False)

        validate_files(fasta_A, HVQ_A, fasta_B, HVQ_B, IGTname)
        # Process alpha
        TRA_HVQ = process_alpha(HVQ_A)
        # Process beta
        TRB_HVQ = process_beta(HVQ_B)

        TRA_HVQ_merged = process_data_TRA(TRA_HVQ, HVQ_A_junction, ['Sequence ID', 'JUNCTION', 'N-REGION'], ['Sequence ID', 'JUNCTION', 'alpha.vj.nregion'])
        TRB_HVQ_merged = process_data_TRB(TRB_HVQ, HVQ_B_junction, ['Sequence ID', 'D-GENE and allele', 'JUNCTION', 'N-REGION', 'N1-REGION', 'N2-REGION'], \
                              ['Sequence ID', 'JUNCTION', 'beta.dj.nregion','beta.vd.nregion' ,'beta.vj.nregion'])

        TRA_merged = process_and_merge_TRA(TRA_HVQ_merged, IGTname)
        TRB_merged = process_and_merge_TRB(TRB_HVQ_merged, IGTname)
        
        merged_final = merge_TRA_TRB(TRA_merged, TRB_merged, IGTname)
        

In [18]:
%%capture output

# Run the process_files function with the specified path
process_files('path_to_folder')

with open(f"{path}CheckRows.txt", "w") as file:
    file.write(output.stdout)